# 04 — K-Means Kümeleme
`rfm_segments.csv` üzerinde Elbow + Silhouette ile optimal k bulunur, ardından k=3 ile kümeleme yapılır.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import os

rfm = pd.read_csv('rfm_segments.csv')
os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'font.family'      : 'DejaVu Serif',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.dpi'       : 150,
})

# Log dönüşümü + StandardScaler
X = rfm[['Recency','Frequency','Monetary']].copy()
X['Frequency'] = np.log1p(X['Frequency'])
X['Monetary']  = np.log1p(X['Monetary'])

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Veri hazır: {X_scaled.shape[0]:,} müşteri, 3 özellik')

Veri hazır: 4,338 müşteri, 3 özellik


In [7]:
# ── ELBOW + SİLHOUETTE ─────────────────────────────────────

inertias    = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))
    print(f'k={k}  |  Inertia: {km.inertia_:>10.1f}  |  Silhouette: {silhouette_score(X_scaled, km.labels_):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Şekil 8. Optimal Küme Sayısı Analizi', fontweight='bold')

axes[0].plot(K_range, inertias, 'o-', color='#2980b9', linewidth=2, markersize=7)
axes[0].set_xlabel('Küme Sayısı (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Dirsek Yöntemi (Elbow Method)')
axes[0].axvline(x=3, color='#e74c3c', linestyle='--', linewidth=1.5, label='Seçilen k=3')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 's-', color='#27ae60', linewidth=2, markersize=7)
axes[1].set_xlabel('Küme Sayısı (k)')
axes[1].set_ylabel('Silhouette Skoru')
axes[1].set_title('Silhouette Analizi')
axes[1].axvline(x=3, color='#e74c3c', linestyle='--', linewidth=1.5, label='Seçilen k=3')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/fig8_elbow_silhouette.png', bbox_inches='tight')
plt.show()

k=2  |  Inertia:     6827.1  |  Silhouette: 0.4077
k=3  |  Inertia:     4253.0  |  Silhouette: 0.4165
k=4  |  Inertia:     3195.3  |  Silhouette: 0.3835
k=5  |  Inertia:     2708.9  |  Silhouette: 0.3455
k=6  |  Inertia:     2346.5  |  Silhouette: 0.3339
k=7  |  Inertia:     2115.6  |  Silhouette: 0.3034
k=8  |  Inertia:     1915.1  |  Silhouette: 0.3015
k=9  |  Inertia:     1770.7  |  Silhouette: 0.2852
k=10  |  Inertia:     1640.4  |  Silhouette: 0.2846


/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_9263/264681903.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ── K-MEANS k=3 ─────────────────────────────────────────────

km = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(X_scaled)

cluster_summary = rfm.groupby('Cluster').agg(
    Müşteri_Sayısı = ('CustomerID',  'count'),
    Ort_Recency    = ('Recency',     'mean'),
    Ort_Frequency  = ('Frequency',   'mean'),
    Ort_Monetary   = ('Monetary',    'mean'),
    Top_Monetary   = ('Monetary',    'sum'),
).round(1)

print('Küme Özeti:')
print(cluster_summary)

# Etiketleme: Recency en düşük + Monetary en yüksek → Platin
label_map = {}
for c in [0, 1, 2]:
    r = cluster_summary.loc[c, 'Ort_Recency']
    m = cluster_summary.loc[c, 'Ort_Monetary']
    if r == cluster_summary['Ort_Recency'].min() and m == cluster_summary['Ort_Monetary'].max():
        label_map[c] = 'Platin Müşteri'
    elif r == cluster_summary['Ort_Recency'].max():
        label_map[c] = 'Gümüş Müşteri'
    else:
        label_map[c] = 'Altın Müşteri'

rfm['Cluster_Label'] = rfm['Cluster'].map(label_map)

total = rfm['Monetary'].sum()
print('\nGelir payları:')
for label, grp in rfm.groupby('Cluster_Label'):
    print(f'  {label:<18}: {grp["CustomerID"].count():>5} müşteri  |  %{grp["Monetary"].sum()/total*100:.1f} gelir')

Küme Özeti:
         Müşteri_Sayısı  Ort_Recency  Ort_Frequency  Ort_Monetary  \
Cluster                                                             
0                  2022         54.4            2.0         577.8   
1                  1335         29.6            9.8        4554.8   
2                   981        254.6            1.4         366.8   

         Top_Monetary  
Cluster                
0           1168221.3  
1           6080624.2  
2            359791.5  

Gelir payları:
  Altın Müşteri     :  2022 müşteri  |  %15.4 gelir
  Gümüş Müşteri     :   981 müşteri  |  %4.7 gelir
  Platin Müşteri    :  1335 müşteri  |  %79.9 gelir


In [4]:
# ── ŞEKİL 9: 3 Boyutlu Kümeleme Grafiği ─────────────────────

COLORS = {
    'Platin Müşteri': '#e74c3c',
    'Altın Müşteri' : '#f39c12',
    'Gümüş Müşteri' : '#95a5a6'
}

fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')

for label, grp in rfm.groupby('Cluster_Label'):
    ax.scatter(
        grp['Recency'],
        np.log1p(grp['Frequency']),
        np.log1p(grp['Monetary']),
        label=label, alpha=0.5, s=20,
        color=COLORS[label], edgecolors='none'
    )

ax.set_xlabel('Recency (Gün)')
ax.set_ylabel('log(Frequency)')
ax.set_zlabel('log(Monetary £)')
ax.set_title('Şekil 9. K-Means Kümeleme Sonucu (3 Boyutlu)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig9_kmeans_3d.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_9263/3438242266.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── ŞEKİL 10: Küme Ortalama RFM Karşılaştırması ─────────────

cluster_stats = rfm.groupby('Cluster_Label').agg(
    Recency   = ('Recency',   'mean'),
    Frequency = ('Frequency', 'mean'),
    Monetary  = ('Monetary',  'mean'),
).round(1)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Şekil 10. Kümelerin Ortalama RFM Değerleri', fontweight='bold')

for ax, (col, ylabel, color) in zip(axes, [
        ('Recency',   'Ort. Recency (Gün)',       '#2980b9'),
        ('Frequency', 'Ort. Frequency (Sipariş)', '#27ae60'),
        ('Monetary',  'Ort. Monetary (£)',         '#e67e22')]):
    vals = cluster_stats[col]
    bars = ax.bar(vals.index, vals.values, color=color, edgecolor='white')
    ax.set_ylabel(ylabel)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + vals.max()*0.01,
                f'{val:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig10_cluster_rfm_karsilastirma.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_9263/3815602244.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── ŞEKİL 11: Büyüklük ve Gelir ─────────────────────────────

seg_ord = rfm.groupby('Cluster_Label').agg(
    Sayi  = ('CustomerID','count'),
    Gelir = ('Monetary','sum')
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Şekil 11. Küme Büyüklüğü ve Toplam Gelir', fontweight='bold')

for ax, col, ylabel in zip(axes, ['Sayi','Gelir'], ['Müşteri Sayısı','Toplam Gelir (£)']):
    bars = ax.bar(seg_ord.index, seg_ord[col],
                  color=[COLORS[l] for l in seg_ord.index], edgecolor='white')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, seg_ord[col]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + seg_ord[col].max()*0.01,
                f'{val:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig11_cluster_buyukluk_gelir.png', bbox_inches='tight')
plt.show()

# Kaydet
rfm.to_csv('rfm_clusters.csv', index=False)
print('\n✅ rfm_clusters.csv kaydedildi')


✅ rfm_clusters.csv kaydedildi


/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_9263/3059052848.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
